In [ ]:
using FFTW
using DifferentialEquations
using Plots

In [ ]:

# KdV: u_t + u u_x + u_xxx = 0 on a periodic domain x in [0, L)
# These defaults are chosen to run quickly in a notebook; increase N/tspan for higher fidelity.
N = 128
L = 2pi
x = L .* (0:N-1) ./ N

# Fourier wave numbers
k = (2pi / L) .* vcat(0:N÷2, -N÷2+1:-1)
ik = 1im .* k
ik3 = ik .^ 3

# 2/3-rule dealiasing mask for nonlinear term
kcut = (2 / 3) * maximum(abs.(k))
dealias = abs.(k) .> kcut

# Integrating-factor formulation in Fourier space
# v̂(t) = exp(ik^3 t) û(t)  ->  dv̂/dt = -exp(ik^3 t) * FFT(u u_x)
function kdv_if_pseudospectral!(dvhat, vhat, p, t)
    ik, ik3, dealias = p

    Eminus = exp.(-ik3 .* t)
    Eplus = exp.(ik3 .* t)
    uhat = Eminus .* vhat

    u = real(ifft(uhat))
    ux = real(ifft(ik .* uhat))

    nlh = fft(u .* ux)
    nlh[dealias] .= 0

    dvhat .= -(Eplus .* nlh)
end


In [ ]:

# Periodic initial condition
u0 = 3.0 .* sech.(0.5 .* (x .- L / 2)) .^ 2
vhat0 = fft(u0)

tspan = (0.0, 4.0)
p = (ik, ik3, dealias)
prob = ODEProblem(kdv_if_pseudospectral!, vhat0, tspan, p)

# DifferentialEquations.jl is used for time stepping.
sol = solve(prob, Tsit5(); reltol=1e-6, abstol=1e-6, saveat=0.05, maxiters=10^8)

In [ ]:

# Reconstruct u(x,t) from integrating-factor variable v̂
Ucols = [real.(ifft(exp.(-ik3 .* t) .* vhat)) for (t, vhat) in zip(sol.t, sol.u)]
U = reduce(hcat, Ucols)  # size (N, nt)

contourf(
    x, sol.t, U',
    xlabel="x", ylabel="t",
    title="KdV solution via pseudospectral method",
    c=:viridis, colorbar=true
    )